<a href="https://colab.research.google.com/github/AlexisSantiago21/CIIC5015-Project2-Neural-Networks/blob/main/regresion_mpg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto II: Redes Neuronales Fully Connected - Regresión
**Estudiantes** Alexis M. Santiago Rivera, Carlos S. Hernandez  
**Curso:** CIIC 5015 - Introducción a la IA  
**Objetivo:** Predecir las millas por galón (MPG) usando diferentes arquitecturas de redes neuronales.

Importaciones:

In [3]:
!pip install torch>=2.0
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print(f"PyTorch Version: {torch.__version__}") # Para confirmar que es 2.0+

PyTorch Version: 2.10.0+cpu


Como nota no deja instalar la 2.0, sale: ERROR: Could not find a version that satisfies the requirement torch==2.0.0+cu118 (from versions: 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0)
ERROR: No matching distribution found for torch==2.0.0+cu118. Codigo usado: !pip install torch==2.0.0+cu118 torchvision==0.15.1+cu118 torchaudio==2.0.1 --extra-index-url https://download.pytorch.org


Carga y Limpieza de Datos:

In [4]:
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

dataset = pd.read_csv(url, names=column_names, na_values='?',
                      comment='\t', sep=' ', skipinitialspace=True).dropna()


dataset['Origin'] = dataset['Origin'].map({1: 'USA', 2: 'Europe', 3: 'Japan'})
dataset = pd.get_dummies(dataset, columns=['Origin'], prefix='', prefix_sep='')
dataset.tail()

,MPG,Cylinders,Displacement,Horsepower,Weight,Acceleration,Model Year,Europe,Japan,USA
393,27.0,4,140.0,86.0,2790.0,15.6,82,False,False,True
394,44.0,4,97.0,52.0,2130.0,24.6,82,True,False,False
395,32.0,4,135.0,84.0,2295.0,11.6,82,False,False,True
396,28.0,4,120.0,79.0,2625.0,18.6,82,False,False,True
397,31.0,4,119.0,82.0,2720.0,19.4,82,False,False,True


Normalización y Tensores:

In [5]:
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42)
train_labels, test_labels = train_df.pop('MPG'), test_df.pop('MPG')

scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(train_df), dtype=torch.float32)
y_train = torch.tensor(train_labels.values, dtype=torch.float32).view(-1, 1)
X_test = torch.tensor(scaler.transform(test_df), dtype=torch.float32)
y_test = torch.tensor(test_labels.values, dtype=torch.float32).view(-1, 1)

Definición de las 3 Redes:

In [6]:
# Network #1: Basic regression
model1 = nn.Sequential(nn.Linear(X_train.shape[1], 1))

# Network #2: 4-layer network
model2 = nn.Sequential(
    nn.Linear(X_train.shape[1], 10), nn.ReLU(),
    nn.Linear(10, 20), nn.ReLU(),
    nn.Linear(20, 10), nn.ReLU(),
    nn.Linear(10, 1)
)

# Network #3: 5-layer network
model3 = nn.Sequential(
    nn.Linear(X_train.shape[1], 10), nn.ReLU(),
    nn.Linear(10, 20), nn.ReLU(),
    nn.Linear(20, 30), nn.ReLU(),
    nn.Linear(30, 20), nn.ReLU(),
    nn.Linear(20, 1)
)

Entrenamiento:

In [7]:
def train(model, epochs=20, lr=0.01):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()
    return loss.item()

print("Entrenando Modelos...")
train(model1); train(model2); train(model3)

Entrenando Modelos...


176.43563842773438

Evaluación y Respuesta:

In [8]:
with torch.no_grad():
    err1 = nn.MSELoss()(model1(X_test), y_test).item()
    err2 = nn.MSELoss()(model2(X_test), y_test).item()
    err3 = nn.MSELoss()(model3(X_test), y_test).item()

print(f"MSE Modelo 1: {err1:.4f}\nMSE Modelo 2: {err2:.4f}\nMSE Modelo 3: {err3:.4f}")

MSE Modelo 1: 549.1282
MSE Modelo 2: 468.7102
MSE Modelo 3: 116.4405


Conclusión: El modelo que presentó el menor error de validación fue el Modelo #3 con un MSE de 116.44. Esto se debe a que su arquitectura de 5 capas permite capturar de manera más efectiva las relaciones no lineales entre los atributos de los vehículos y su eficiencia de combustible (MPG), superando significativamente a la red básica de una sola neurona.